# End-to-End GEM Reconstruction and Analysis: *E. coli* K-12

Demonstrates the full `gsoc-sysbio-llm-tools` pipeline:
```
CarveMe → MEMOTE → COBRApy
```
All steps call the running MCP servers via HTTP using a correct MCP JSON-RPC session.

**Prerequisites:** `docker-compose up --build` must be running before executing this notebook.

> **Note:** This notebook runs on your **host machine**. `carve`, `memote`, and `cobra`
> are only installed *inside* the Docker containers. Never call them directly from here —
> always go through the MCP sessions below.

## Cell 1 — Imports and configuration

In [ ]:
import requests
import subprocess
import json
import time
import os

# MCP server URLs — matching docker-compose host port mapping
CARVEME = 'http://localhost:8003/mcp'
MEMOTE  = 'http://localhost:8002/mcp'
COBRAPY = 'http://localhost:8001/mcp'
ORCH    = 'http://localhost:8080'

ORGANISM_ID = 'ecoli_k12'
SBML_PATH   = f'/models/{ORGANISM_ID}/model.xml'   # path inside containers

def pp(d):
    print(json.dumps(d, indent=2, default=str))

## Cell 2 — MCP session helper

FastMCP uses **Streamable HTTP transport** — not plain REST.
Every interaction requires a 3-step flow:

```
1. POST /mcp   method=initialize              → server returns Mcp-Session-Id header
2. POST /mcp   method=notifications/initialized  (no id — it is a notification)
3. POST /mcp   method=tools/call              → actual tool call
```

Every request must carry:
- `Content-Type: application/json`
- `Accept: application/json, text/event-stream`
- `Mcp-Session-Id: <id from step 1>` (steps 2 and 3 only)

The response may be `application/json` **or** `text/event-stream` (SSE) — we handle both.

In [ ]:
_HEADERS = {
    'Content-Type': 'application/json',
    'Accept': 'application/json, text/event-stream',
}

class MCPSession:
    """Minimal synchronous MCP client for notebook use."""

    def __init__(self, base_url: str):
        self.base_url  = base_url
        self.session_id = None
        self._req_id   = 0
        self._s        = requests.Session()
        self._init()

    # ── internals ────────────────────────────────────────────────────────────

    def _nid(self):
        self._req_id += 1
        return self._req_id

    def _headers(self):
        h = dict(_HEADERS)
        if self.session_id:
            h['Mcp-Session-Id'] = self.session_id
        return h

    def _post(self, payload):
        return self._s.post(self.base_url, json=payload,
                            headers=self._headers(), timeout=900)

    def _init(self):
        # Step 1: initialize — capture session ID from response headers
        r = self._post({
            'jsonrpc': '2.0', 'id': self._nid(), 'method': 'initialize',
            'params': {
                'protocolVersion': '2025-03-26',
                'capabilities': {},
                'clientInfo': {'name': 'notebook', 'version': '1.0'},
            }
        })
        r.raise_for_status()
        self.session_id = r.headers.get('mcp-session-id')

        # Step 2: notifications/initialized — no 'id' (it is a notification, not a request)
        n = self._post({'jsonrpc': '2.0', 'method': 'notifications/initialized'})
        assert n.status_code in (200, 202), \
            f'Init notification failed: {n.status_code} {n.text}'

    def _parse(self, resp):
        """Handle both application/json and text/event-stream responses."""
        ct = resp.headers.get('content-type', '')
        if 'text/event-stream' in ct:
            for line in resp.text.splitlines():
                if line.startswith('data:'):
                    try:
                        msg = json.loads(line[5:].strip())
                        if 'result' in msg or 'error' in msg:
                            return msg
                    except json.JSONDecodeError:
                        pass
            raise RuntimeError(f'No result in SSE stream:\n{resp.text[:500]}')
        return resp.json()

    # ── public API ───────────────────────────────────────────────────────────

    def call_tool(self, name: str, arguments: dict) -> dict:
        resp = self._post({
            'jsonrpc': '2.0', 'id': self._nid(), 'method': 'tools/call',
            'params': {'name': name, 'arguments': arguments},
        })
        resp.raise_for_status()
        msg = self._parse(resp)

        if 'error' in msg:
            raise RuntimeError(f"Tool '{name}' error: {msg['error']}")

        # FastMCP wraps the return value in result.content[0].text as a JSON string
        content = msg.get('result', {}).get('content', [])
        if content and content[0].get('type') == 'text':
            try:
                return json.loads(content[0]['text'])
            except (json.JSONDecodeError, KeyError):
                return {'raw': content[0].get('text')}
        return msg.get('result', {})

print('MCPSession ready.')

## Cell 3 — Verify all services are up

In [ ]:
# Orchestrator health (plain REST — no MCP session needed)
print('Orchestrator:', requests.get(f'{ORCH}/health').json())

# Verify MCP session init works for each server
for name, url in [('carveme', CARVEME), ('memote', MEMOTE), ('cobrapy', COBRAPY)]:
    try:
        s = MCPSession(url)
        print(f'{name:10s} OK  session_id={s.session_id}')
    except Exception as e:
        print(f'{name:10s} FAILED — {e}')

## Cell 4 — Debug: inspect CarveMe inside its container

`carve` is only installed **inside** the `sysbio-carveme` Docker container.
Use `docker exec` to inspect it — never call it directly from the host.

In [ ]:
def docker_exec(container: str, cmd: list[str]) -> str:
    """Run a command inside a running container and return stdout+stderr."""
    proc = subprocess.run(
        ['docker', 'exec', container] + cmd,
        capture_output=True, text=True, timeout=30
    )
    return (proc.stdout + proc.stderr).strip()

# Check carve is available and see its actual flags
print('=== carve --help ===')
print(docker_exec('sysbio-carveme', ['carve', '--help']))

# Check diamond is available
print('\n=== diamond version ===')
print(docker_exec('sysbio-carveme', ['diamond', '--version']))

# Check the /models directory is mounted correctly inside the container
print('\n=== /models contents ===')
print(docker_exec('sysbio-carveme', ['ls', '-la', '/models']))

## Cell 5 — Step 1: Reconstruct *E. coli* K-12 with CarveMe

Uses RefSeq accession `GCF_000005845.2` (MG1655 complete genome).
CarveMe downloads and annotates the genome automatically (~10 min).

In [ ]:
carveme = MCPSession(CARVEME)

# check_carveme_install runs inside the container — confirms carve is on PATH
pp(carveme.call_tool('check_carveme_install', {}))

In [ ]:
print('Reconstructing E. coli K-12 (this takes 5-15 min)...')
result = carveme.call_tool('carve_from_refseq', {
    'refseq_id':   'GCF_000005845.2',
    'output_path': SBML_PATH,
    'gram':        'neg',
})
pp(result)
assert 'error' not in result, (
    f"CarveMe failed.\n"
    f"cmd:    {result.get('cmd')}\n"
    f"stderr: {result.get('stderr')}\n"
    f"stdout: {result.get('stdout')}"
)
print(f'\nModel written to: {result["sbml_path"]}')

## Cell 6 — Step 2: Quality check with MEMOTE

A fresh CarveMe model typically scores 0.5–0.65.
Main gaps are annotation (ChEBI/KEGG IDs). Consistency should be high.

In [ ]:
print('Running MEMOTE (2-5 min)...')
memote  = MCPSession(MEMOTE)
quality = memote.call_tool('get_memote_summary', {'sbml_path': SBML_PATH})

print(f"Overall score : {quality.get('total_score')}")
print(f"Passed        : {quality.get('passed_count')}")
print(f"Warned        : {quality.get('warned_count')}")
print(f"Failed        : {quality.get('failed_count')}")
print('\nSection scores:')
for k, v in quality.get('section_scores', {}).items():
    print(f'  {k}: {v}')
print('\nTop 5 failures:')
for t in quality.get('failed_tests', [])[:5]:
    print(f"  • {t['title']}: {t['summary']}")

## Cell 7 — Step 3: Load into COBRApy and set growth medium

Expected aerobic growth on glucose minimal medium: ~0.874 h⁻¹

In [ ]:
cobra = MCPSession(COBRAPY)

load = cobra.call_tool('load_model', {
    'sbml_path': SBML_PATH,
    'model_id':  ORGANISM_ID,
})
print('Loaded:', load)

# Inspect exchange reactions before setting medium
exchanges = cobra.call_tool('get_exchange_reactions', {'model_id': ORGANISM_ID})
print(f'\nExchange reactions: {len(exchanges)}')
# Show glucose/oxygen/nitrogen/phosphate ones
for ex in exchanges:
    if any(k in ex['id'] for k in ['glc', 'o2', 'nh4', 'pi', 'so4']):
        print(f"  {ex['id']:25s} lb={ex['lower_bound']:6.1f}  ub={ex['upper_bound']:6.1f}  {ex['name']}")

In [ ]:
# Aerobic glucose minimal medium
aerobic_medium = {
    'EX_glc__D_e': 10.0,
    'EX_o2_e':     20.0,
    'EX_nh4_e':    10.0,
    'EX_pi_e':     10.0,
    'EX_so4_e':    10.0,
    'EX_k_e':      10.0,
}

aerobic = cobra.call_tool('set_media', {
    'model_id': ORGANISM_ID,
    'media':    aerobic_medium,
})
print(f"Aerobic growth:   {aerobic['fba_growth_rate']:.4f} h⁻¹  (status: {aerobic['fba_status']})")

# Anaerobic: remove oxygen
anaerobic = cobra.call_tool('set_media', {
    'model_id': ORGANISM_ID,
    'media':    {**aerobic_medium, 'EX_o2_e': 0.0},
})
print(f"Anaerobic growth: {anaerobic['fba_growth_rate']:.4f} h⁻¹  (status: {anaerobic['fba_status']})")
print('\nAerobic > anaerobic — expected for E. coli.')

# Restore aerobic for subsequent steps
cobra.call_tool('set_media', {'model_id': ORGANISM_ID, 'media': aerobic_medium})
print('Medium reset to aerobic.')

## Cell 8 — Step 4: Essential gene identification

Expect ~15–25% of genes to be essential in minimal medium.
Reference: ~300 essential genes in the iJO1366 E. coli model.

⚠️ This can take 5–20 min on a large model.

In [ ]:
print('Running single-gene deletion screen...')
ess = cobra.call_tool('essential_genes', {'model_id': ORGANISM_ID})

pct = ess['essential_count'] / ess['total_genes'] if ess['total_genes'] else 0
print(f"Total genes:   {ess['total_genes']}")
print(f"Essential:     {ess['essential_count']}  ({pct:.1%})")
print(f"Non-essential: {ess['nonessential_count']}")
print(f"\nFirst 20: {ess['essential'][:20]}")

## Cell 9 — Step 5: Flux Variability Analysis (FVA)

FVA reveals which reactions are fixed vs flexible under optimal growth.
We focus on glycolysis here as a targeted example.

In [ ]:
glycolysis = ['PGI', 'PFK', 'FBA', 'TPI', 'GAPD', 'PGK', 'PGM', 'ENO', 'PYK']

fva = cobra.call_tool('run_fva', {
    'model_id':            ORGANISM_ID,
    'reaction_list':       glycolysis,
    'fraction_of_optimum': 0.9,
})

print(f"FVA at {fva['fraction_of_optimum']*100:.0f}% of optimal growth:")
print(f"{'Reaction':<12} {'Min':>10} {'Max':>10} {'Range':>10}")
print('-' * 46)
for r in fva['results']:
    print(f"{r['id']:<12} {r['minimum']:>10.4f} {r['maximum']:>10.4f} {r['range']:>10.4f}")

## Cell 10 — One-shot via orchestrator

The orchestrator runs the full CarveMe → MEMOTE → COBRApy pipeline
as a background job. Same result as cells 5–8 in one call.

In [ ]:
resp = requests.post(f'{ORCH}/workflow/reconstruct-and-analyze', json={
    'organism_id': ORGANISM_ID,
    'refseq_id':   'GCF_000005845.2',
    'gram':        'neg',
    'aerobic':     True,
}).json()
print('Job started:', resp)
job_id = resp['job_id']

# Poll until done
while True:
    status = requests.get(f'{ORCH}/workflow/status/{job_id}').json()
    print(f"  {status['status']:12s} step={status.get('step')}")
    if status['status'] in ('completed', 'failed'):
        break
    time.sleep(30)

print('\nFinal result:')
pp(status)

## Summary

| Step | Tool | Expected result |
|------|------|-----------------|
| 1. Reconstruct | CarveMe | SBML at `/models/ecoli_k12/model.xml` |
| 2. Quality     | MEMOTE  | Score ~0.55–0.65, annotation gaps flagged |
| 3. FBA         | COBRApy | Aerobic ~0.874 h⁻¹, anaerobic lower |
| 4. Essentiality | COBRApy | ~15–25% genes essential |
| 5. FVA         | COBRApy | Glycolytic flux ranges |

### Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `FileNotFoundError: carve` | Running `carve` on host, not in container | Use `docker exec` or MCP session |
| `404 /tools/call` | Wrong MCP endpoint | Endpoint is `/mcp`, method is in JSON body |
| `Bad Request: Missing session ID` | No init handshake | Use `MCPSession` — handles init automatically |
| CarveMe produces no SBML | `/models` dir not mounted or `-o` flag issue | Check `docker-compose volumes`, use updated `server.py` |